# Practical Application III: Comparing Classifiers

**Overview**: In this practical application, your goal is to compare the performance of the classifiers we encountered in this section, namely K Nearest Neighbor, Logistic Regression, Decision Trees, and Support Vector Machines.  We will utilize a dataset related to marketing bank products over the telephone.  



### Getting Started

Our dataset comes from the UCI Machine Learning repository [link](https://archive.ics.uci.edu/ml/datasets/bank+marketing).  The data is from a Portugese banking institution and is a collection of the results of multiple marketing campaigns.  We will make use of the article accompanying the dataset [here](CRISP-DM-BANK.pdf) for more information on the data and features.



### Problem 1: Understanding the Data

To gain a better understanding of the data, please read the information provided in the UCI link above, and examine the **Materials and Methods** section of the paper.  How many marketing campaigns does this data represent?

In [35]:
"""
The dataset collected is related to 17 campaigns that occurred between May 2008 and November 2010,
corresponding to a total of 79354 contacts.For the whole database considered, there were 6499
successes (8% success rate).
"""

'\nThe dataset collected is related to 17 campaigns that occurred between May 2008 and November 2010,\ncorresponding to a total of 79354 contacts.For the whole database considered, there were 6499\nsuccesses (8% success rate).\n'

### Problem 2: Read in the Data

Use pandas to read in the dataset `bank-additional-full.csv` and assign to a meaningful variable name.

In [36]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.dummy import DummyClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC

from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split, GridSearchCV

from sklearn.metrics import accuracy_score, classification_report, f1_score, confusion_matrix, roc_auc_score, make_scorer

In [37]:
df = pd.read_csv('data/bank-additional-full.csv', sep = ';')

In [38]:
df.head()

,age,job,marital,education,default,housing,loan,contact,month,day_of_week,...,campaign,pdays,previous,poutcome,emp.var.rate,cons.price.idx,cons.conf.idx,euribor3m,nr.employed,y
0,56,housemaid,married,basic.4y,no,no,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
1,57,services,married,high.school,unknown,no,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
2,37,services,married,high.school,no,yes,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
3,40,admin.,married,basic.6y,no,no,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
4,56,services,married,high.school,no,no,yes,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no


### Problem 3: Understanding the Features
Examine the data description below, and determine if any of the features are missing values or need to be coerced to a different data type.

### Input variables:
#### bank client data:
1 - age (numeric)  
2 - job : type of job (categorical: 'admin.','blue-collar','entrepreneur','housemaid','management','retired','self-employed','services','student','technician','unemployed','unknown')  
3 - marital : marital status (categorical: 'divorced','married','single','unknown'; note: 'divorced' means divorced or widowed)  
4 - education (categorical: 'basic.4y','basic.6y','basic.9y','high.school','illiterate','professional.course','university.degree','unknown')  
5 - default: has credit in default? (categorical: 'no','yes','unknown')  
6 - housing: has housing loan? (categorical: 'no','yes','unknown')  
7 - loan: has personal loan? (categorical: 'no','yes','unknown')  
    
#### related with the last contact of the current campaign:
8 - contact: contact communication type (categorical: 'cellular','telephone')  
9 - month: last contact month of year (categorical: 'jan', 'feb', 'mar', ..., 'nov', 'dec')  
10 - day_of_week: last contact day of the week (categorical: 'mon','tue','wed','thu','fri')  
11 - duration: last contact duration, in seconds (numeric). Important note: this attribute highly affects the output target (e.g., if duration=0 then y='no'). Yet, the duration is not known before a call is performed. Also, after the end of the call y is obviously known. Thus, this input should only be included for benchmark purposes and should be discarded if the intention is to have a realistic predictive model.  

#### other attributes:
12 - campaign: number of contacts performed during this campaign and for this client (numeric, includes last contact)        
13 - pdays: number of days that passed by after the client was last contacted from a previous campaign (numeric; 999 means client was not previously contacted)  
14 - previous: number of contacts performed before this campaign and for this client (numeric)  
15 - poutcome: outcome of the previous marketing campaign (categorical: 'failure','nonexistent','success')  

#### social and economic context attributes  
16 - emp.var.rate: employment variation rate - quarterly indicator (numeric)  
17 - cons.price.idx: consumer price index - monthly indicator (numeric)  
18 - cons.conf.idx: consumer confidence index - monthly indicator (numeric)  
19 - euribor3m: euribor 3 month rate - daily indicator (numeric)  
20 - nr.employed: number of employees - quarterly indicator (numeric)  

#### Output variable (desired target):
21 - y - has the client subscribed a term deposit? (binary: 'yes','no')  

In [61]:
df.info()
print(df.shape)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 41188 entries, 0 to 41187
Data columns (total 21 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   age             41188 non-null  int64  
 1   job             41188 non-null  object 
 2   marital         41188 non-null  object 
 3   education       41188 non-null  object 
 4   default         41188 non-null  object 
 5   housing         41188 non-null  object 
 6   loan            41188 non-null  object 
 7   contact         41188 non-null  object 
 8   month           41188 non-null  object 
 9   day_of_week     41188 non-null  object 
 10  duration        41188 non-null  int64  
 11  campaign        41188 non-null  int64  
 12  pdays           41188 non-null  int64  
 13  previous        41188 non-null  int64  
 14  poutcome        41188 non-null  object 
 15  emp.var.rate    41188 non-null  float64
 16  cons.price.idx  41188 non-null  float64
 17  cons.conf.idx   41188 non-null 

### Problem 4: Understanding the Task

After examining the description and data, your goal now is to clearly state the *Business Objective* of the task.  State the objective below.

In [62]:
"""
# BUSINESS OBJECTIVE
The classification goal is to predict if the client will subscribe a term deposit (variable y). 

"""

'\n# BUSINESS OBJECTIVE\nThe classification goal is to predict if the client will subscribe a term deposit (variable y). \n\n'

### Problem 5: Engineering Features

Now that you understand your business objective, we will build a basic model to get started.  Before we can do this, we must work to encode the data.  Using just the bank information features, prepare the features and target column for modeling with appropriate encoding and transformations.

In [41]:
# Variable selection and feature engineering
# Exclluding variable in column 11 (duration) as it is highly correlated target column.  I also wanted to create a realistic model instead of a benchmark model. 
df_prep = df.drop(['duration'], axis=1)
print(df_prep.shape)

X = df_prep.drop(['y'], axis=1)
y = df_prep['y'] 

(41188, 20)


In [42]:
# Since we're not missing any values, we're going to leave out SimpleImputer for the numeric_cols. 
num_cols = ['age', 'campaign', 'pdays', 'previous', 'emp.var.rate', 'cons.price.idx', 'cons.conf.idx', 'euribor3m', 'nr.employed']
cat_cols = ['job','marital','education','default', 'housing', 'loan', 'contact', 'month', 'day_of_week']

preprocesor = ColumnTransformer(
    transformers = [
        ('num', StandardScaler(), num_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore'), cat_cols)
    ]
)

### Problem 6: Train/Test Split

With your data prepared, split it into a train and test set.

In [43]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

### Problem 7: A Baseline Model

Before we build our first model, we want to establish a baseline.  What is the baseline performance that our classifier should aim to beat?

In [44]:
# First we'll use a DummyClassifier to establish a baseline accuracyi.  This is the baseline performance that our classifier will need to beat. 
dummy_clf = DummyClassifier().fit(X_train, y_train)
baseline_accuracy = dummy_clf.score(X_test, y_test)
print(f"Baseline accuracy: {baseline_accuracy:.4f}")


Baseline accuracy: 0.8874


### Problem 8: A Simple Model

Use Logistic Regression to build a basic model on your data.  

In [ ]:
# Now we will fit the four models without any hypterparameter tuning and see how they perform. 
# Logistic Regression
baseline_lr = Pipeline([
    ('preprocessor', preprocesor),
    ('classifier', LogisticRegression())
])
baseline_lr.fit(X_train, y_train)
y_pred_lr = baseline_lr.predict(X_test)
y_train_pred_lr = baseline_lr.predict(X_train)
lr_accuracy = accuracy_score(y_test, y_pred_lr)
lr_train_accuracy = accuracy_score(y_train, y_train_pred_lr)
lr_f1 = f1_score(y_test, y_pred_lr, pos_label='yes')

Logistic Regression test accuracy: 0.9007
Logistic Regression train accuracy: 0.8998
Logistic Regression F1 score: 0.3317


In [ ]:
# KNN first
baseline_knn = Pipeline([
    ('preprocessor', preprocesor),
    ('classifier', KNeighborsClassifier())
])
baseline_knn.fit(X_train, y_train)
y_pred_knn = baseline_knn.predict(X_test)
y_train_pred_knn = baseline_knn.predict(X_train)
knn_accuracy = accuracy_score(y_test, y_pred_knn)
knn_train_accuracy = accuracy_score(y_train, y_train_pred_knn)
knn_f1 = f1_score(y_test, y_pred_knn, pos_label='yes')

KNN test accuracy: 0.8967
KNN train accuracy: 0.9120
KNN F1 score: 0.3969


In [ ]:
baseline_dt = Pipeline([
    ('preprocessor', preprocesor),
    ('classifier', DecisionTreeClassifier(random_state=42))
])
y_pred_dt = baseline_dt.fit(X_train, y_train).predict(X_test)
y_train_pred_dt = baseline_dt.predict(X_train)
dt_accuracy = accuracy_score(y_test, y_pred_dt)
dt_train_accuracy = accuracy_score(y_train, y_train_pred_dt)
dt_f1 = f1_score(y_test, y_pred_dt, pos_label='yes')

Decision Tree test accuracy: 0.8395
Decision Tree train accuracy: 0.9954
Decision Tree F1 score: 0.3227


In [ ]:
baseline_svm = Pipeline([
    ('preprocessor', preprocesor),
    ('classifier', SVC(random_state=42))
])
y_pred_svm = baseline_svm.fit(X_train, y_train).predict(X_test)
y_train_pred_svm = baseline_svm.predict(X_train)
svm_accuracy = accuracy_score(y_test, y_pred_svm)
svm_train_accuracy = accuracy_score(y_train, y_train_pred_svm)
svm_f1 = f1_score(y_test, y_pred_svm, pos_label='yes')

SVM test accuracy: 0.9031
SVM train accuracy: 0.9049
SVM F1 score: 0.3687


### Problem 9: Score the Model

What is the accuracy of your model?

In [60]:
print(f"Logistic Regression test accuracy: {lr_accuracy:.4f}")
print(f"Logistic Regression train accuracy: {lr_train_accuracy:.4f}")
print(f"Logistic Regression F1 score: {lr_f1:.4f}")

print(f"KNN test accuracy: {knn_accuracy:.4f}")
print(f"KNN train accuracy: {knn_train_accuracy:.4f}")
print(f"KNN F1 score: {knn_f1:.4f}")

print(f"Decision Tree test accuracy: {dt_accuracy:.4f}")
print(f"Decision Tree train accuracy: {dt_train_accuracy:.4f}")
print(f"Decision Tree F1 score: {dt_f1:.4f}")

print(f"SVM test accuracy: {svm_accuracy:.4f}")
print(f"SVM train accuracy: {svm_train_accuracy:.4f}")
print(f"SVM F1 score: {svm_f1:.4f}")


Logistic Regression test accuracy: 0.9007
Logistic Regression train accuracy: 0.8998
Logistic Regression F1 score: 0.3317
KNN test accuracy: 0.8967
KNN train accuracy: 0.9120
KNN F1 score: 0.3969
Decision Tree test accuracy: 0.8395
Decision Tree train accuracy: 0.9954
Decision Tree F1 score: 0.3227
SVM test accuracy: 0.9031
SVM train accuracy: 0.9049
SVM F1 score: 0.3687


### Problem 10: Model Comparisons

Now, we aim to compare the performance of the Logistic Regression model to our KNN algorithm, Decision Tree, and SVM models.  Using the default settings for each of the models, fit and score each.  Also, be sure to compare the fit time of each of the models.  Present your findings in a `DataFrame` similar to that below:

| Model | Train Time | Train Accuracy | Test Accuracy | F1 Score | 
| ----- | ---------- | -------------  | -----------   | -------- | 
| dummy classifier    |  0.0  |0.8874     |.     |     |
| LogisticRegression | 0.2s | 0.8998 | 0.9007 | 0.3317 | 
| KNN | 0.5s | 0.9120 | 0.8967 | 0.3969 | 
| Decision Tree | 0.3s | 0.9954 | 0.8395 | 0.3227 | 
| SVM | 1m 13.2s | 0.9049 | 0.9031 | 0.3687 | 

In [63]:
# KNN and Decision Tree are overfitting the training data, as there is a large gap between train/test accuracy.  
# Logistic Regression and SVM are performing better on the test set. So far SVM is on top. 

In [ ]:
# Next, I'll use GridSearchCV to tune the hyperparameters of the models and see if we can improve the performance.

### Problem 11: Improving the Model

Now that we have some basic models on the board, we want to try to improve these.  Below, we list a few things to explore in this pursuit.


- Hyperparameter tuning and grid search.  All of our models have additional hyperparameters to tune and explore.  For example the number of neighbors in KNN or the maximum depth of a Decision Tree.  
- Adjust your performance metric

In [74]:
# Let's add some scaling with StandardScaler. 
lr_tuned = Pipeline([
    ('preprocessor', preprocesor),
    ('scaler', StandardScaler()),
    ('classifier', LogisticRegression())
])
params = {'classifier__C': [0.01, 0.1, 1, 10, 100]}
f1_yes = make_scorer(f1_score, pos_label='yes')
grid_search_lr = GridSearchCV(lr_tuned, params, cv=5, scoring=f1_yes)
grid_search_lr.fit(X_train, y_train, groups=None)
best_lr = grid_search_lr.best_estimator_
print("Best C:", grid_search_lr.best_params_['classifier__C'])

lr_tuned.set_params(**grid_search_lr.best_params_)
lr_tuned.fit(X_train, y_train)
y_pred_lr = lr_tuned.predict(X_test)
print("LR Accuracy:", accuracy_score(y_test, y_pred_lr))
print("LR f1_score:", f1_score(y_test, y_pred_lr, pos_label='yes'))
print("LR Classification Report:\n", classification_report(y_test, y_pred_lr))    


Best C: 10
LR Accuracy: 0.9011896091284293
LR f1_score: 0.3414239482200647
LR Classification Report:
               precision    recall  f1-score   support

          no       0.91      0.99      0.95      7310
         yes       0.69      0.23      0.34       928

    accuracy                           0.90      8238
   macro avg       0.80      0.61      0.64      8238
weighted avg       0.88      0.90      0.88      8238



In [76]:
# Let's try to tune KNN with GridSearchCV
knn_tuned = Pipeline([
    ('preprocessor', preprocesor),
    ('scaler', StandardScaler()),
    ('classifier', KNeighborsClassifier())
])
params = {'classifier__n_neighbors': [1, 3, 5, 7, 9, 11, 13, 15]}
f1_yes = make_scorer(f1_score, pos_label='yes')
grid_search = GridSearchCV(knn_tuned, param_grid=params, cv=5, scoring=f1_yes)
grid_search.fit(X_train, y_train, groups=None)
print("Best K:", grid_search.best_params_)
knn_tuned.set_params(**grid_search.best_params_)

knn_tuned.fit(X_train, y_train)
y_pred_knn = knn_tuned.predict(X_test)
print("KNN Accuracy:", accuracy_score(y_test, y_pred_knn))
print("f1_score:", f1_score(y_test, y_pred_knn, pos_label='yes'))
print("KNN Classification Report:\n", classification_report(y_test, y_pred_knn))    

Best K: {'classifier__n_neighbors': 5}
KNN Accuracy: 0.892571012381646
f1_score: 0.3572984749455338
KNN Classification Report:
               precision    recall  f1-score   support

          no       0.91      0.97      0.94      7310
         yes       0.55      0.27      0.36       928

    accuracy                           0.89      8238
   macro avg       0.73      0.62      0.65      8238
weighted avg       0.87      0.89      0.88      8238



In [77]:
#Tuning up DecisionTreeClassifier
dt_tuned = Pipeline([
    ('preprocessor', preprocesor),
    ('classifier', DecisionTreeClassifier(random_state=42))
])
params = {'classifier__max_depth': [None, 5, 10, 15, 20]}
f1_yes = make_scorer(f1_score, pos_label='yes')
grid_search = GridSearchCV(dt_tuned, param_grid=params, cv=5, scoring=f1_yes)
grid_search.fit(X_train, y_train, groups=None)
print("Best max_depth:", grid_search.best_params_)
dt_tuned.set_params(**grid_search.best_params_) 
dt_tuned.fit(X_train, y_train)
y_pred_dt = dt_tuned.predict(X_test)
print("Decision Tree Accuracy:", accuracy_score(y_test, y_pred_dt))
print("Decision Tree f1_score:", f1_score(y_test, y_pred_dt, pos_label='yes'))
print("Decision Tree Classification Report:\n", classification_report(y_test, y_pred_dt))

Best max_depth: {'classifier__max_depth': 10}
Decision Tree Accuracy: 0.9002184996358339
Decision Tree f1_score: 0.38839285714285715
Decision Tree Classification Report:
               precision    recall  f1-score   support

          no       0.91      0.98      0.95      7310
         yes       0.63      0.28      0.39       928

    accuracy                           0.90      8238
   macro avg       0.77      0.63      0.67      8238
weighted avg       0.88      0.90      0.88      8238



In [ ]:
"""
# SVM tuning with GridSearchCV
svm_tuned = Pipeline([
    ('preprocessor', preprocesor),
    ('scaler', StandardScaler()),
    ('classifier', SVC(random_state=42))
])
params = {'classifier__C': [0.1, 1, 10], 'classifier__kernel': ['linear', 'rbf']}
f1_yes = make_scorer(f1_score, pos_label='yes')
grid_search = GridSearchCV(svm_tuned, param_grid=params, cv=5, scoring=f1_yes)
grid_search.fit(X_train, y_train, groups=None)
print("Best params:", grid_search.best_params_)
svm_tuned.set_params(**grid_search.best_params_)
svm_tuned.fit(X_train, y_train)
y_pred_svm = svm_tuned.predict(X_test)
print("SVM Accuracy:", accuracy_score(y_test, y_pred_svm))
print("SVM f1_score:", f1_score(y_test, y_pred_svm, pos_label='yes'))
print("SVM Classification Report:\n", classification_report(y_test, y_pred_svm))
"""

In [ ]:
# SVM tuning with GridSearchCV, trying with 2 C values and one classifier kernel.  This is to see if we can improve the performance of the SVM model. Also trying cv=3
# Also trying parallelized processing with n_jobs=-1 to speed up the process.
svm_tuned = Pipeline([
    ('preprocessor', preprocesor),
    ('scaler', StandardScaler()),
    ('classifier', SVC(random_state=42))
])
params = {'classifier__C': [0.1, 1], 'classifier__kernel': [ 'rbf']}
f1_yes = make_scorer(f1_score, pos_label='yes')
grid_search = GridSearchCV(svm_tuned, param_grid=params, cv=3, scoring=f1_yes, n_jobs=-1)
grid_search.fit(X_train, y_train, groups=None)
print("Best params:", grid_search.best_params_)
svm_tuned.set_params(**grid_search.best_params_)
svm_tuned.fit(X_train, y_train)
y_pred_svm = svm_tuned.predict(X_test)
print("SVM Accuracy:", accuracy_score(y_test, y_pred_svm))
print("SVM f1_score:", f1_score(y_test, y_pred_svm, pos_label='yes'))
print("SVM Classification Report:\n", classification_report(y_test, y_pred_svm))

##### Questions